# Age and Gender Distortion

In a [recent paper in Nature](https://www.nature.com/articles/s41586-025-09581-z), Douglas Guilbeault, Solène Delecourt & Bhargav Srinivasa Desikan investigated the effects of age distortion on genders.

In this assignent, you will work through some of their results yoursefl. You will use their data, available at <https://github.com/drguilbe/distortion_age_gender_online/>.

## Setup

**Python.** Use Python 3.10+. Create a virtual environment and install dependencies so anyone can rerun the notebook:

```bash
python3 -m venv .venv
source .venv/bin/activate 
pip install -r requirements.txt
```

**Packages** (see `requirements.txt`): `pandas`, `numpy`, `matplotlib`, `seaborn`, `statsmodels`, `scipy`, `pingouin`, `scipy.stats`, `plotly`, `jupyter`, `ipykernel`.


## Part 1 — Correlation between age and gender (GPT-2 Large)

We load `GPT2-large-dimensions.csv` (one row per social category). The file includes **three extraction methods** for each construct, following the paper’s robustness checks:

| Construct | Columns (raw scores, not min–max normalized) |
|-----------|-----------------------------------------------|
| Age | `age.main`, `age.ext`, `age.red` |
| Gender | `gender.main`, `gender.ext`, `gender.red` |

The **primary** Pearson correlation matches the replication R script (`fig2_GPT2-large_analyses.R`): **`age.main`** vs **`gender.main`**. Companion columns `*_norm.*` are min–max normalized versions used elsewhere (e.g. regression); we use the raw `age.*` / `gender.*` triples for the main correlation and for the robustness heatmaps.

On the **heatmaps**, the main method is labeled **`age_score`** / **`gender_score`** (same values as `age.main` / `gender.main`). Axes follow **`red`, `ext`, `score`** order to match the assignment figures.

The summary table uses **Pingouin**’s `corr` (Pearson *r*, 95% CI, *p*, Bayes factor BF10, post-hoc power) so the printed output aligns with the assignment example.

In [3]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pingouin as pg
import re
import seaborn as sns

# Notebook directory (works in Jupyter)
BASE = Path.cwd()
DATA_PATH = BASE / "GPT2-large-dimensions.csv"

df = pd.read_csv(DATA_PATH)
assert len(df) == 3495, "Expected 3,495 social categories (check CSV version)."


def raw_dimension_columns(frame: pd.DataFrame, family: str) -> list[str]:
    """Return ordered raw age.* or gender.* columns (main, ext, red) for robustness heatmaps."""
    pat = re.compile(rf"^{re.escape(family)}\.(main|ext|red)$")
    order = ("main", "ext", "red")
    cols = [c for c in frame.columns if pat.match(c)]
    return sorted(cols, key=lambda c: order.index(c.split(".")[1]))


AGE_COLS = raw_dimension_columns(df, "age")
GENDER_COLS = raw_dimension_columns(df, "gender")

print("Age dimension columns:", AGE_COLS)
print("Gender dimension columns:", GENDER_COLS)
print("Alternate / normalized columns (examples):", [c for c in df.columns if "norm" in c.lower()][:6], "...")

# Primary correlation (matches authors' R: age.main vs gender.main)
MAIN_AGE, MAIN_GENDER = "age.main", "gender.main"
pearson_tbl = pg.corr(df[MAIN_AGE], df[MAIN_GENDER], method="pearson")
pearson_tbl = pearson_tbl.rename(columns={"p_val": "p-val", "CI95": "CI95%"})

pearson_tbl

Age dimension columns: ['age.main', 'age.ext', 'age.red']
Gender dimension columns: ['gender.main', 'gender.ext', 'gender.red']
Alternate / normalized columns (examples): ['gender_norm.main', 'gender_norm.ext', 'gender_norm.red', 'age_norm.main', 'age_norm.ext', 'age_norm.red'] ...


,n,r,CI95%,p-val,BF10,power
pearson,3495,0.872559,"[0.86, 0.88]",0.0,inf,1.0


In [4]:
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

# Heatmap palette: low *r* base → light coral → medium red → darkest red
_CORR_CMAP_COLORS = [
    "#E48568",
    "#F08070",
    "#E86F5A",
    "#E05B4A",
    "#D84A3A",
    "#D1002F",
    "#C8002E",
]


def correlation_colormap() -> LinearSegmentedColormap:
    return LinearSegmentedColormap.from_list("corr_red", _CORR_CMAP_COLORS, N=256)


# Color scale and legend ticks (match assignment reference: 0.60 … 1.00 in steps of 0.05)
_COLORBAR_VMIN = 0.6
_COLORBAR_VMAX = 1.0
_COLORBAR_TICKS = np.linspace(_COLORBAR_VMIN, _COLORBAR_VMAX, 9)


def pairwise_corr_matrix(
    frame: pd.DataFrame,
    *,
    cols: list[str],
    labels: list[str],
) -> pd.DataFrame:
    """Pearson *r* matrix; ``cols`` order defines row/column order on the figure."""
    sub = frame[cols].dropna()
    r = sub.corr(method="pearson").reindex(index=cols, columns=cols)
    r.index = labels
    r.columns = labels
    return r


def plot_correlation_heatmap(
    corr: pd.DataFrame,
    *,
    out_path: Path,
    vmin: float = _COLORBAR_VMIN,
    vmax: float = _COLORBAR_VMAX,
    figsize: tuple[float, float] = (4.5, 3.8),
) -> None:
    """Custom palette; color bar fixed 0.60–1.00 with ticks every 0.05 (assignment reference)."""
    vmin_val, vmax_val = vmin, vmax
    cmap = correlation_colormap()

    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(
        corr,
        annot=False,
        vmin=vmin_val,
        vmax=vmax_val,
        cmap=cmap,
        square=True,
        linewidths=0.5,
        linecolor="white",
        ax=ax,
        cbar_kws={
            "shrink": 0.85,
            "ticks": _COLORBAR_TICKS,
            "format": "%.2f",
        },
    )
    ax.set_title("Correlation Heatmap", fontsize=12)

    # White labels on all cells (matches assignment figures, including low-*r* coral cells)
    for i in range(corr.shape[0]):
        for j in range(corr.shape[1]):
            val = float(corr.iloc[i, j])
            ax.text(
                j + 0.5,
                i + 0.5,
                f"{val:.2f}",
                ha="center",
                va="center",
                color="#FFFFFF",
                fontsize=10,
            )

    plt.tight_layout()
    fig.savefig(out_path, format="svg", bbox_inches="tight")
    plt.close(fig)


# Axis order and *_score labels match the assignment figures (main = .main in CSV)
age_r = pairwise_corr_matrix(
    df,
    cols=["age.red", "age.ext", "age.main"],
    labels=["age_red", "age_ext", "age_score"],
)
gender_r = pairwise_corr_matrix(
    df,
    cols=["gender.red", "gender.ext", "gender.main"],
    labels=["gender_red", "gender_ext", "gender_score"],
)

plot_correlation_heatmap(age_r, out_path=BASE / "correlation_heatmap_age.svg")
plot_correlation_heatmap(gender_r, out_path=BASE / "correlation_heatmap_gender.svg")


print("Saved:", BASE / "correlation_heatmap_age.svg")
print("Saved:", BASE / "correlation_heatmap_gender.svg")

Saved: /Users/annamegalou/Downloads/age_gender_distortion/correlation_heatmap_age.svg
Saved: /Users/annamegalou/Downloads/age_gender_distortion/correlation_heatmap_gender.svg
